# Error Analysis


## Setup & Imports

Set `EXPERIMENT_NAME` below to the arm you want to analyse (the non-pseudonymized or the pseudonymized run). Run this notebook after the Evaluation notebook, so `experiment.jsonl` is already collapsed to the matched set and `similarities.npz` exists for that arm. If you re-run on an experiment whose data has changed, delete its `error_analysis_*.jsonl` caches first so they rebuild.

In [ ]:
# Import libraries
import json
from pathlib import Path
import pandas as pd

# Set up key paths & constants
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROMPTS_DIR = ROOT / "models" / "llama" / "prompts"
EXPERIMENT_NAME = "experiment_test_9385_20260609_1013"                  # change it to experiment of your choice
EXPERIMENT_DIR  = ROOT / "data" / "experiments" / EXPERIMENT_NAME
LLAMA_RUNS  = EXPERIMENT_DIR / "llama_runs"
EXPERIMENT  = EXPERIMENT_DIR / "experiment.jsonl"

# Set up display for better displays
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", None)

# Confirm the root, it should point out to main repo folder
print(f"ROOT = {ROOT}")

### Analysis 1 — Structural confusion: do wrong retrievals share a plot skeleton?

**Goal:** When the structural model retrieves the wrong story, is it because of a random error/noise, or whether the errors are meaningful where these "wrong" stories are structurally similar (e.g., share same archetypical event trigger and/or event type skeleton)? 

**Method:**
1. Take every query whose top-1 retrieval belongs to a different work, so the actual errors hapenned. Call the wrongly retrieved summary "wrong match". 
2. For each (query, wrong match) pair, measure how much narrative structure the two share, in four ways: 
    1. Shared *event types* via Traversky Index overlap: How much *event types* overlap between query and the wrong match?
    2. Longest in-order common event type sequence (namely, LCS ratio) for *event types*: What is the longest sequence of *event types* that appear in both the query and the wrong match? 
    3. Shared *event triggers* via Traversky Index overlap: How much *event triggers* overlap between query and the wrong match?
    4. Longest in-order common event type sequence (namely, LCS ratio) for *event triggers*: What is the longest sequence of *event triggers* that appear in both the query and the wrong match?
3. Build a chance baseline: pair the same queries with a random story (also a different work) and compute the same four measures. These are the ratios created, as if we would do things randomly.
4. Compare error pairs vs random pairs (one-sided Mann–Whitney U): if the scores of match between (query, wrong match) is higher than the scores of match (query, random match), the errors are systematic structural confusion (i.e., signal)
5. Visually inspect the top pairs by hand: the shared event sequence (the candidate plot skeleton), genres, languages, and the two texts side by side.

**Results:**
- 77.5% of queries (4,393 / 5,666) retrieve a wrong story at rank 1 (Qwen3 × events_only).
- The wrong story shares **almost twice** the event categories with the query that a random story would (Dice 0.44 vs 0.23).
- The same holds for **order**: the shared in-order event sequence is ~2× longer than chance (0.20 vs 0.11) — so it's plot *shape*, not just shared ingredients.
- Even on **exact trigger verbs**, the wrong match is ~3× above chance (0.10 vs 0.04) — so the effect is not an artifact of coarse or noisy event-type labels.
- All four differences are significant at p ≈ 0. Sanity check: the random baseline for event-type overlap (0.23) reproduces the corpus-wide mean from pipeline §8.8 / Table 5 (0.237).
- Top confusions share interpretable skeletons across languages and genres — e.g. *Sending → Arriving → Giving → Warning* shared by a German doctor drama and an Italian mafia film with **zero** shared genres.

In [ ]:
import random
import numpy as np
from IPython.display import display, Markdown
from scipy.stats import mannwhitneyu

# All conditions share the same events, but each has a different similarity matrix, so the rank-1 wrong match changes per cell.
#   views: qwen3_emb_0p6b | e5_mistral
#   conditions: events_only | temporal | causal | temporal_causal_independent | temporal_causal_joint
PRIMARY_VIEW, PRIMARY_COND = "qwen3_emb_0p6b", "events_only"
ALL_VIEWS = ["qwen3_emb_0p6b", "e5_mistral"]
ALL_CONDS = ["events_only", "temporal", "causal", "temporal_causal_independent", "temporal_causal_joint"]
SEED = 0    # Tweak this to change the randomness in this analysis, yet keep it same for reproduciblility.

# Read from caches instead of re-loading the full data each run, to save time.
META_CACHE   = EXPERIMENT_DIR / "error_analysis_meta.jsonl"
SIMILARITIES = EXPERIMENT_DIR / "similarities.npz"

# 1. Per-row metadata only, since experiment.jsonl carries the embeddings (~4.4 GB).
if not META_CACHE.exists():
    with open(EXPERIMENT, encoding="utf-8") as f, open(META_CACHE, "w", encoding="utf-8") as out:
        for line in f:
            r = json.loads(line)
            out.write(json.dumps({
                "wikidata_id": r["wikidata_id"], "summary_id": r["summary_id"],
                "lang": r.get("lang"), "genres": r.get("genres") or [],
                "triggers": [e["trigger"].lower() for e in r.get("events") or []],
                "types":    [e["event_type"]      for e in r.get("events") or []],
                "text": r.get("text", ""),
            }, ensure_ascii=False) + "\n")
meta  = [json.loads(l) for l in META_CACHE.read_text(encoding="utf-8").splitlines() if l.strip()]
works = np.array([m["wikidata_id"] for m in meta])
N     = len(meta)

# 2. Skeleton-overlap measures. Inventory = set-based Dice; order = LCS ratio with the same normalization
#    (2*LCS/(|a|+|b|)), so inventory and order numbers are directly comparable.
def dice(a, b):
    A, B = set(a), set(b)
    return 2 * len(A & B) / (len(A) + len(B)) if (A or B) else 0.0

# Longest-common-subsequence DP table: table[i][j] = LCS length of a[:i] and b[:j]. Reused both for the
# order ratio below and to recover the shared skeleton itself in the 1.3 visual check.
def lcs_table(a, b):
    L = [[0] * (len(b) + 1) for _ in range(len(a) + 1)]
    for i, x in enumerate(a):
        for j, y in enumerate(b):
            L[i + 1][j + 1] = L[i][j] + 1 if x == y else max(L[i][j + 1], L[i + 1][j])
    return L

# Order-aware overlap: 2*LCS/(|a|+|b|), the order-respecting analogue of Dice (same normalization, so the
# two are directly comparable).
def lcs_ratio(a, b):
    return 2 * lcs_table(a, b)[-1][-1] / (len(a) + len(b)) if (a or b) else 0.0

MEASURES = {
    "Event-type inventory overlap": lambda u, v: dice(u["types"], v["types"]),
    "Event-type order overlap":     lambda u, v: lcs_ratio(u["types"], v["types"]),
    "Trigger inventory overlap":    lambda u, v: dice(u["triggers"], v["triggers"]),
    "Trigger order overlap":        lambda u, v: lcs_ratio(u["triggers"], v["triggers"]),
}

# 3. For one (encoder, condition): the rank-1 wrong matches, a matched random null, and the per-measure
#    overlap scores. The random null pairs each error query with a random summary of a DIFFERENT work, so it
#    shows what the overlap looks like when the pairing carries no signal. (similarities.npz rows follow
#    experiment.jsonl order; the diagonal is -inf.)
def analyze_cell(view, cond):
    sim = np.load(SIMILARITIES)[f"{view}__{cond}"]
    assert sim.shape == (N, N), f"similarities {sim.shape} vs meta rows {N}; cache stale? delete {META_CACHE}"
    rank1 = sim.argmax(axis=1)
    error_pairs = [(int(query), int(rank1[query])) for query in np.flatnonzero(works[rank1] != works)]
    rng = random.Random(SEED)
    null_pairs = []
    for query, _ in error_pairs:
        candidate = rng.randrange(N)
        while candidate == query or works[candidate] == works[query]:
            candidate = rng.randrange(N)
        null_pairs.append((query, candidate))
    scores = {}
    for name, measure_fn in MEASURES.items():
        error_scores = np.array([measure_fn(meta[query], meta[candidate]) for query, candidate in error_pairs])
        null_scores  = np.array([measure_fn(meta[query], meta[candidate]) for query, candidate in null_pairs])
        scores[name] = (error_scores, null_scores)
    return sim, error_pairs, null_pairs, scores

# 4. Full grid: every encoder x condition. Each cell has its own similarity matrix (hence its own error set
#    and wrong matches); the measures and the per-query random null are built identically (same SEED).
#    Takes several minutes (LCS over ~9k pairs per grid cell).
grid_rows = []
for view in ALL_VIEWS:
    for cond in ALL_CONDS:
        _, error_pairs_cell, _, scores_cell = analyze_cell(view, cond)
        wrong_share = f"{len(error_pairs_cell):,}/{N:,} ({len(error_pairs_cell) / N:.1%})"
        for name, (error_scores, null_scores) in scores_cell.items():
            _, p = mannwhitneyu(error_scores, null_scores, alternative="greater")
            grid_rows.append({"Encoder": view, "Condition": cond, "Wrong@1": wrong_share, "Measure": name,
                              "Error pair mean":  f"{error_scores.mean():.4f}",
                              "Random-pair mean": f"{null_scores.mean():.4f}",
                              "Δ":            f"{error_scores.mean() - null_scores.mean():+.4f}",
                              "p":                "< .001" if p < .001 else f"{p:.3f}"})
display(pd.DataFrame(grid_rows).set_index(["Encoder", "Condition", "Wrong@1", "Measure"]))

# 5. Keep the primary cell's results in module scope (normally, we had other related analysis but now removed.)
VIEW, COND = PRIMARY_VIEW, PRIMARY_COND
sim, error_pairs, null_pairs, scores = analyze_cell(VIEW, COND)

### 1.3 Visually check the cases

**Goal**: Qualitatively spot check random cases to see exemplars visually

In [ ]:
from IPython.display import display, Markdown

TOP_K   = 3      # Tweak this to see the Top K amounf of results
PREVIEW = None   # chars of raw text shown per summary; None = full text (for the thesis appendix)

# Truncate a summary's text for display (full text when PREVIEW is None).
def truncate(text):
    return text if PREVIEW is None or len(text) <= PREVIEW else text[:PREVIEW] + "…"

# Backtrack the LCS DP table to recover one shared subsequence (the shared skeleton itself).
def lcs_sequence(a, b):
    table = lcs_table(a, b)
    i, j, sequence = len(a), len(b), []
    while i and j:
        if a[i-1] == b[j-1]:
            sequence.append(a[i-1]); i -= 1; j -= 1
        elif table[i-1][j] >= table[i][j-1]:
            i -= 1
        else:
            j -= 1
    return sequence[::-1]

# Rank the primary cell's error pairs by event-type order overlap and show the TOP_K most confused.
error_order_overlap = scores["Event-type order overlap"][0]
ranked_by_overlap   = np.argsort(-error_order_overlap)

# Define title for markdown
lines = [f"## Top {TOP_K} structural confusions under `{VIEW}__{COND}` (by event-type order overlap)\n"]

# Print each information related to event type/trigger overlap with full texts to compare
for rank, pair_idx in enumerate(ranked_by_overlap[:TOP_K], 1):
    query, candidate = error_pairs[pair_idx]
    query_meta, candidate_meta = meta[query], meta[candidate]
    shared_skeleton = lcs_sequence(query_meta["types"], candidate_meta["types"])
    shared_genres   = sorted(set(query_meta["genres"]) & set(candidate_meta["genres"]))
    lines.append(
        f"### {rank}. `{query_meta['wikidata_id']}__{query_meta['summary_id']}` → `{candidate_meta['wikidata_id']}__{candidate_meta['summary_id']}`"
        f" &nbsp; (cosine similarity {sim[query, candidate]:.3f}, event type order overlap {error_order_overlap[pair_idx]:.3f}, "
        f"event type inventory {dice(query_meta['types'], candidate_meta['types']):.3f}, event trigger inventory overlap {dice(query_meta['triggers'], candidate_meta['triggers']):.3f})\n\n"
        f"- **shared event-type skeleton ({len(shared_skeleton)} types)**: {' → '.join(shared_skeleton)}\n"
        f"- **shared genres**: {', '.join(shared_genres) if shared_genres else '(none)'}"
        f" &nbsp;|&nbsp; langs: {query_meta['lang']} vs {candidate_meta['lang']}\n\n"
        f"- **Event Types of the Query ({len(query_meta['types'])})**: {' → '.join(query_meta['types'])}\n"
        f"- **Event Types of the Wrong Match ({len(candidate_meta['types'])})**: {' → '.join(candidate_meta['types'])}\n"
        f"- **Event Triggers of the Query:** {', '.join(query_meta['triggers'])}\n"
        f"- **Event Triggers of the Wrong Match**: {', '.join(candidate_meta['triggers'])}\n"
        f"- **Text of the Query**: {truncate(query_meta['text'])}\n"
        f"- **Text of the Wrong Match**: {truncate(candidate_meta['text'])}\n"
    )
display(Markdown("\n".join(lines)))

### Analysis 2 — Structural mismatch: why does adding relations make retrieval worse?

**Goal:** Adding temporal/causal relations consistently *lowers* retrieval performance relative to events-only, so we find the mechanism: does the relation layer add discriminative signal, or does it just make all stories look more alike?

**What the relation layer actually adds.** Relations are linearized in the hybrid format, e.g. *(e2:left|Departing, BEFORE, e1:arrived|Arriving)* — endpoints carry trigger and type. So, relative to the events-only string, the relation block adds three things:
- (a) **Relation labels** (e.g., *BEFORE*, *CAUSE*, ...), drawn from a taxonomies from literature
- (b) **Repeated copies of event tokens** the *EVENTS:* block already contains: an event linked in 10 relations gets its trigger/type repeated 10 times, re-weighting events by how often Llama links them, not by narrative importance
- (c) **Actual relation pattern** (which event pairs connect) — the only genuinely new information, and only useful if it is not predictable from event order

**Method:**
1. **Collapse check:** Mean and SD of the pairwise cosine similarity across all summaries, per condition. If the mean rises and the SD shrinks as relations are added, the relation layer pushes all stories closer together and compresses the spread that retrieval depends on.
2. **Label diversity:** The distribution of relation labels per condition. If one label (e.g. *BEFORE*) dominates, the labels cannot discriminate between stories.
3. **Link-pattern predictability:**  The share of relations connecting adjacent events (*e_i → e_(i+1)*). If most links are adjacent-pair chains, the link pattern is derivable from event order and adds nothing new.
4. **Casualties:** Queries where events-only retrieved the right story at rank 1 but a relation condition put a wrong story on top; inspect whether the wrong match wins on relation-pattern similarity despite different events.


### 2.1 Collapse check
**Goal:** Show whether each added relation layer pushes *all* summaries closer together in embedding space (mean pairwise similarity up, spread down), instead of separating related from unrelated ones.

In [ ]:
# Mean / SD / min / max of pairwise cosine similarity per encoder × condition, straight from similarities.npz.
# Rising mean + shrinking SD as relations are added = the space collapses, which means stories that are different are more likely to be inferred as the same.
similarity_matrices = np.load(SIMILARITIES)
table_rows = []
for view in ["qwen3_emb_0p6b", "e5_mistral"]:
    for cond in ["events_only", "temporal", "causal", "temporal_causal_independent", "temporal_causal_joint"]:
        similarity = similarity_matrices[f"{view}__{cond}"]
        off_diagonal = similarity[np.isfinite(similarity)]            # diagonal is -inf (self-similarity masked)
        table_rows.append({"Encoder": view, "Condition": cond,
                           "Mean pairwise cosine": f"{off_diagonal.mean():.4f}",
                           "SD": f"{off_diagonal.std():.4f}",
                           "Min": f"{off_diagonal.min():.4f}",
                           "Max": f"{off_diagonal.max():.4f}"})
display(pd.DataFrame(table_rows).set_index(["Encoder", "Condition"]))

### 2.2 Relation-label diversity
**Goal:** Check whether the relation labels are diverse enough to distinguish stories, or whether one generic label (e.g. *BEFORE*) dominates everything.

In [ ]:
from collections import Counter

# Relation-label distribution per Llama condition: how often each relation label is emitted. A few labels
# dominating (low diversity) is part of why relation enrichment does not help retrieval.
REL_CACHE   = EXPERIMENT_DIR / "error_analysis_relations.jsonl"
LLAMA_CONDS = ["temporal", "causal", "temporal_causal_independent", "temporal_causal_joint"]

# Cache just the relation triples per condition (experiment.jsonl also carries the embeddings, which we skip).
if not REL_CACHE.exists():
    with open(EXPERIMENT, encoding="utf-8") as infile, open(REL_CACHE, "w", encoding="utf-8") as outfile:
        for line in infile:
            summary = json.loads(line)
            record = {"wikidata_id": summary["wikidata_id"], "summary_id": summary["summary_id"], "conditions": {}}
            for cond in LLAMA_CONDS:
                condition_relations = (summary.get("conditions", {}).get(cond) or {}).get("relations") or {}
                record["conditions"][cond] = [[triple["source"], triple["relation"], triple["target"]]
                                              for triples in condition_relations.values() for triple in (triples or [])]
            outfile.write(json.dumps(record) + "\n")
rels = [json.loads(l) for l in REL_CACHE.read_text(encoding="utf-8").splitlines() if l.strip()]
assert len(rels) == N, f"relations cache has {len(rels)} rows vs meta {N}; delete {REL_CACHE} and re-run"

# Count label frequencies per condition.
table_rows = []
for cond in LLAMA_CONDS:
    label_counts = Counter(label for record in rels for _, label, _ in record["conditions"][cond])
    total_labels = sum(label_counts.values())
    for label, count in label_counts.most_common():
        table_rows.append({"Condition": cond, "Label": label, "Count": count, "Share": f"{count / total_labels:.1%}"})
display(pd.DataFrame(table_rows).set_index(["Condition", "Label"]))

### 2.3 Link-pattern predictability
**Goal:** Check whether the relations mostly link adjacent events. If so, the link pattern is derivable from event order and carries almost no new information.

In [ ]:
# Share of relations that just link adjacent events (e_i -> e_(i+1)), per condition.
# Uses the relations cache built in 2.2.
table_rows = []
for cond in LLAMA_CONDS:
    adjacent = forward = total_relations = 0
    for record in rels:
        for source, _, target in record["conditions"][cond]:
            source_idx, target_idx = int(source[1:]), int(target[1:])
            total_relations += 1
            forward  += source_idx < target_idx
            adjacent += abs(target_idx - source_idx) == 1
    table_rows.append({"Condition": cond, "n relations": total_relations,
                       "% adjacent (|target − source| = 1)": f"{adjacent / total_relations:.1%}",
                       "% forward (source < target)":        f"{forward / total_relations:.1%}"})
display(pd.DataFrame(table_rows).set_index("Condition"))

### 2.4 Casualties (right → wrong)
**Goal:** Count the queries that events-only retrieved correctly but a relation condition broke — the direct damage the relation layer causes.

In [ ]:
# Per (encoder, relation condition): how many queries events_only answered correctly at rank 1
# become wrong ("broken"), how many wrong ones become right ("fixed"), and the net effect.
# Run for BOTH embedders so we can compare this across architectures.
similarity_matrices = np.load(SIMILARITIES)
BASELINE_COND = "events_only"
ALL_VIEWS  = ["qwen3_emb_0p6b", "e5_mistral"]
REL_CONDS  = ["temporal", "causal", "temporal_causal_independent", "temporal_causal_joint"]

table_rows, broken_examples, baseline_correct_count = [], {}, {}
for view in ALL_VIEWS:
    baseline_rank1   = similarity_matrices[f"{view}__{BASELINE_COND}"].argmax(axis=1)
    baseline_correct = works[baseline_rank1] == works
    baseline_correct_count[view] = int(baseline_correct.sum())
    for cond in REL_CONDS:
        relation_rank1 = similarity_matrices[f"{view}__{cond}"].argmax(axis=1)
        broken_idx = np.flatnonzero(baseline_correct  & (works[relation_rank1] != works))
        fixed_idx  = np.flatnonzero(~baseline_correct & (works[relation_rank1] == works))
        broken_examples[(view, cond)] = [(int(query), int(baseline_rank1[query]), int(relation_rank1[query])) for query in broken_idx]
        table_rows.append({"Encoder": view, "Condition": cond,
                           "Broken (right → wrong)": len(broken_idx),
                           "Fixed (wrong → right)":  len(fixed_idx),
                           "Net": len(fixed_idx) - len(broken_idx)})
display(pd.DataFrame(table_rows).set_index(["Encoder", "Condition"]))

### 2.4b Broken-retrieval examples

**Goal:** Inspect a few queries that events_only retrieved correctly but a relation condition broke, showing the correct story against the wrong story the relation condition preferred (with event-type inventory overlap for context). Reuses `broken_examples` from the table above.

In [ ]:
# A few broken examples: the correct story events_only found vs the wrong story the relation
# condition preferred, with event-type overlap for context.
EXAMPLE_VIEW, EXAMPLE_COND, N_EXAMPLES = VIEW, "temporal_causal_joint", 3
lines = [f"#### Broken examples under `{EXAMPLE_VIEW}__{EXAMPLE_COND}` (events_only was right)\n"]
for query, correct_match, wrong_match in broken_examples[(EXAMPLE_VIEW, EXAMPLE_COND)][:N_EXAMPLES]:
    query_meta, correct_meta, wrong_meta = meta[query], meta[correct_match], meta[wrong_match]
    lines.append(
        f"- query `{query_meta['wikidata_id']}__{query_meta['summary_id']}`: events_only → correct "
        f"`{correct_meta['wikidata_id']}__{correct_meta['summary_id']}` (type inventory {dice(query_meta['types'], correct_meta['types']):.2f}); "
        f"{EXAMPLE_COND} → wrong `{wrong_meta['wikidata_id']}__{wrong_meta['summary_id']}` "
        f"(type inventory {dice(query_meta['types'], wrong_meta['types']):.2f})"
    )
display(Markdown("\n".join(lines)))

### Analysis 3 — Annotation noise budget

**Goal:** Quantify everything the pipeline removed or silently lost between the raw test TMA subset and the evaluated corpus, so the surviving noise is bounded, citable, and cannot silently explain the retrieval results.

**Method:**
1. **Pipeline funnel:** how many summaries each stage removed, and why.
2. **Llama failure modes:** split the §4.5 drop into JSON parse errors vs context-window overflows, plus rows that hit the generation token cap but still parsed (silently truncated relation lists).
3. **Silently dropped labels:** schema-valid relations whose label is outside the codebook (e.g. *AFTER*), dropped at validation — counted per condition and per source language (UNI-76).
4. **Hallucinated event IDs:** relations pointing at non-existent events, dropped in §4.6 (UNI-60) — rate and the *e(N+1)* pattern.
5. **Relation density & truncation:** relations per event, and whether token-cap rows have thinner relation lists.
6. **Residual trigger noise:** linking-verb triggers (~1%, retained) and subword fragments (fixed, expect ≈ 0).

**Results**


### 3.1 Pipeline funnel
**Goal:** One table of how many summaries each pipeline stage removed, and why, from the Section 2 Subseting down to the fully evaluated corpus

In [ ]:
# Summary counts per pipeline stage, with each stage's drop split by cause. Everything is recomputed from
# the stage files themselves (Section 5.5 from experiment.yaml). The Section 4.5 split needs the per-summary
# Llama outcome, so the llama_runs noise cache is built HERE in one streaming pass over
# llama_runs/<cond>.jsonl; 3.2 to 3.4 reuse it.
import re
import yaml

MIN_EVENTS, MIN_PER_WORK = 5, 2   # pipeline Section 3.5: MIN_EVENTS_FOR_RELATIONS, MIN_DEDUPED_EN_SUMMARIES

NOISE_CACHE = EXPERIMENT_DIR / "error_analysis_llama_noise.jsonl"
RAW_CONDS   = ["temporal", "causal", "temporal_causal_joint"]
RAW_KEY     = {"temporal": "temporal_relations", "causal": "causal_relations",
               "temporal_causal_joint": "joint_relations"}
ALLOWED = {cond: set(yaml.safe_load((PROMPTS_DIR / f"{cond}.yaml").read_text())["allowed_labels"])
           for cond in RAW_CONDS}
EID_RE  = re.compile(r"^e\d+$")

# The stored `relations` are the VALIDATED triples, so dropped ones are recovered by re-parsing the verbatim
# `response_raw` audit trail (models/llama/inference.py).
if not NOISE_CACHE.exists():
    with open(NOISE_CACHE, "w", encoding="utf-8") as outfile:
        for cond in RAW_CONDS:
            with open(LLAMA_RUNS / f"{cond}.jsonl", encoding="utf-8") as infile:
                for line in infile:
                    summary = json.loads(line)
                    condition_block = summary["condition_block"]
                    record = {"condition": cond, "wikidata_id": summary["wikidata_id"],
                              "summary_id": summary["summary_id"], "lang": summary.get("lang"),
                              "n_events": len(summary.get("events") or []),
                              "overflow": condition_block.get("source") == "skipped_ctx_overflow",
                              "parse_error": condition_block.get("parse_error") is not None,
                              "hit_ctx_cap": bool(condition_block.get("hit_ctx_cap")),
                              "n_kept": len((condition_block.get("relations") or {}).get(RAW_KEY[cond]) or []),
                              "dropped_labels": {}, "n_drop_malformed": 0, "n_drop_eid": 0}
                    if not record["overflow"] and not record["parse_error"]:
                        try:
                            parsed  = json.loads(condition_block.get("response_raw") or "")
                            triples = parsed.get(RAW_KEY[cond]) if isinstance(parsed, dict) else None
                        except Exception:
                            triples = None
                        invalid_labels = Counter()
                        for triple in triples or []:
                            if not isinstance(triple, dict) or not all(k in triple for k in ("relation", "source", "target")):
                                record["n_drop_malformed"] += 1
                            elif triple["relation"] not in ALLOWED[cond]:
                                invalid_labels[str(triple["relation"])] += 1
                            elif not (isinstance(triple["source"], str) and EID_RE.match(triple["source"])
                                      and isinstance(triple["target"], str) and EID_RE.match(triple["target"])):
                                record["n_drop_eid"] += 1
                        record["dropped_labels"] = dict(invalid_labels)
                    outfile.write(json.dumps(record, ensure_ascii=False) + "\n")
noise = [json.loads(l) for l in open(NOISE_CACHE, encoding="utf-8")]

# Section 3.5 split: fewer than MIN_EVENTS events vs the cluster floor (from the Section 3 output file).
n_extracted, n_too_few_events, work_summary_counts = 0, 0, Counter()
for line in open(EXPERIMENT_DIR / "tma_subset_events_full.test.jsonl", encoding="utf-8"):
    summary = json.loads(line)
    n_extracted += 1
    if len(summary.get("events") or []) >= MIN_EVENTS:
        work_summary_counts[summary["wikidata_id"]] += 1
    else:
        n_too_few_events += 1
dropped_floor_3_5 = sum(count for count in work_summary_counts.values() if count < MIN_PER_WORK)
n_processed       = n_extracted - n_too_few_events - dropped_floor_3_5
assert n_processed == sum(1 for _ in open(EXPERIMENT_DIR / "tma_subset_events_processed.test.jsonl", encoding="utf-8"))

# Section 4.5 split: Llama complete-case vs the cluster floor (from the noise cache).
failed_summaries = {(record["wikidata_id"], record["summary_id"]) for record in noise if record["parse_error"] or record["overflow"]}
survivors_per_work = Counter()
for record in noise:
    if record["condition"] == RAW_CONDS[0] and (record["wikidata_id"], record["summary_id"]) not in failed_summaries:
        survivors_per_work[record["wikidata_id"]] += 1
dropped_floor_4_5 = sum(count for count in survivors_per_work.values() if count < MIN_PER_WORK)

# Section 5.5 split, pre-counted in experiment.yaml.
truncation_stats = yaml.safe_load((EXPERIMENT_DIR / "experiment.yaml").read_text())["pre_embedding_e5_truncation_filter"]
assert n_processed - len(failed_summaries) - dropped_floor_4_5 == truncation_stats["summaries_in"]
# experiment.yaml records the pipeline output (Section 5.5 summaries_out). After it, the Evaluation notebook
# collapses the corpus to the matched intersection I' (summaries in BOTH arms + cluster floor); N reflects
# that final evaluated set. summaries_out == N only when the matched intersection has not been run.
n_matched_drop = truncation_stats["summaries_out"] - N
assert n_matched_drop >= 0, "experiment.jsonl is smaller than expected vs the Section 5.5 output; check N / experiment.yaml"

as_pct = lambda n, base: f"{n:,} ({n / base:.1%})"

# Matched-intersection drop (Evaluation notebook), split into the both-arms intersection and the subsequent
# cluster floor, recomputed from the pre-intersection snapshots saved in BOTH arms.
n_intersection_drop = n_intersection_floor = None
pre_intersection_nonanon = EXPERIMENT_DIR / "experiment_before_anon_intersection.jsonl"
pre_intersection_anon    = EXPERIMENT_DIR.parent / ("anon_" + EXPERIMENT_DIR.name) / "experiment_before_anon_intersection.jsonl"
if n_matched_drop and pre_intersection_nonanon.exists() and pre_intersection_anon.exists():
    id_pattern = re.compile(r'"wikidata_id":\s*"([^"]*)",\s*"summary_id":\s*"([^"]*)"')
    def read_id_set(path):
        keys = set()
        for line in open(path, encoding="utf-8"):
            match = id_pattern.search(line[:160])
            if match: keys.add((match.group(1), match.group(2)))
        return keys
    nonanon_keys, anon_keys = read_id_set(pre_intersection_nonanon), read_id_set(pre_intersection_anon)
    shared_keys = nonanon_keys & anon_keys
    summaries_per_work = Counter(work_id for work_id, _ in shared_keys)
    matched_keys = {(work_id, summary_id) for (work_id, summary_id) in shared_keys if summaries_per_work[work_id] >= MIN_PER_WORK}
    n_intersection_drop, n_intersection_floor = len(nonanon_keys) - len(shared_keys), len(shared_keys) - len(matched_keys)
    assert len(nonanon_keys) == truncation_stats["summaries_out"] and len(matched_keys) == N, "matched-intersection split inputs disagree with the Section 5.5 output / N"

# Each row carries the running corpus size AFTER its own drop, so "Summaries out" is filled throughout.
funnel_rows = [
    {"Stage": "Section 2 subsetting (output)", "Cause": "official test split; genre, translation-dedup, length, BERT-subword, >=2-summaries-per-work filters",
     "Dropped (% of stage input)": "entry stage", "Summaries out": n_extracted},
    {"Stage": "Section 3.5 pre-annotation", "Cause": f"fewer than {MIN_EVENTS} detected events (structural-informativeness floor)",
     "Dropped (% of stage input)": as_pct(n_too_few_events, n_extracted), "Summaries out": n_extracted - n_too_few_events},
    {"Stage": "Section 3.5 pre-annotation", "Cause": f"cluster floor (work left with < {MIN_PER_WORK} summaries)",
     "Dropped (% of stage input)": as_pct(dropped_floor_3_5, n_extracted), "Summaries out": n_processed},
    {"Stage": "Section 4.5 complete-case", "Cause": "Llama relations missing in >=1 condition (parse error / ctx overflow, split in 3.2)",
     "Dropped (% of stage input)": as_pct(len(failed_summaries), n_processed), "Summaries out": n_processed - len(failed_summaries)},
    {"Stage": "Section 4.5 complete-case", "Cause": f"cluster floor (work left with < {MIN_PER_WORK} summaries)",
     "Dropped (% of stage input)": as_pct(dropped_floor_4_5, n_processed), "Summaries out": truncation_stats["summaries_in"]},
    {"Stage": "Section 5.5 linearization cap", "Cause": f"linearized string over {truncation_stats['cap_tokens']} tokens in any condition",
     "Dropped (% of stage input)": as_pct(truncation_stats["summaries_dropped_overflow"], truncation_stats["summaries_in"]), "Summaries out": truncation_stats["summaries_in"] - truncation_stats["summaries_dropped_overflow"]},
    {"Stage": "Section 5.5 linearization cap", "Cause": f"cluster floor (work left with < {MIN_PER_WORK} summaries)",
     "Dropped (% of stage input)": as_pct(truncation_stats["summaries_dropped_new_singletons"], truncation_stats["summaries_in"]), "Summaries out": truncation_stats["summaries_out"]},
]
# Matched-intersection stage (Evaluation notebook): split when the anon arm's snapshot is available, else combined.
if n_matched_drop:
    base_count = truncation_stats["summaries_out"]
    if n_intersection_drop is not None:
        funnel_rows.append(
            {"Stage": "Evaluation: matched intersection", "Cause": "kept only summaries surviving in BOTH the non-anon and anon arms",
             "Dropped (% of stage input)": as_pct(n_intersection_drop, base_count), "Summaries out": base_count - n_intersection_drop})
        funnel_rows.append(
            {"Stage": "Evaluation: matched intersection", "Cause": f"cluster floor (work left with < {MIN_PER_WORK} summaries)",
             "Dropped (% of stage input)": as_pct(n_intersection_floor, base_count), "Summaries out": N})
    else:
        funnel_rows.append(
            {"Stage": "Evaluation: matched intersection", "Cause": "kept only summaries surviving in BOTH arms (intersection + >=2-summaries-per-work floor)",
             "Dropped (% of stage input)": as_pct(n_matched_drop, base_count), "Summaries out": N})
funnel = pd.DataFrame(funnel_rows)
display(funnel.set_index(["Stage", "Cause"]))
print(f"Retained end-to-end: {N:,} / {n_extracted:,} summaries ({N / n_extracted:.1%}); "
      f"percentages are relative to each stage's input.")

### 3.2 Llama failure modes
**Goal:** Split the Section 4.5 drop into its causes — JSON parse errors vs context-window overflows — and count the rows whose generation hit the token cap yet still parsed (silently truncated relation lists).

In [ ]:
# Failure modes per condition, from the noise cache built in 3.1.
table_rows = []
for cond in RAW_CONDS:
    condition_records = [record for record in noise if record["condition"] == cond]
    table_rows.append({"Condition": cond, "Summaries": len(condition_records),
                       "Parse errors": sum(record["parse_error"] for record in condition_records),
                       "Context overflows": sum(record["overflow"] for record in condition_records),
                       "Hit token cap but parsed": sum(record["hit_ctx_cap"] and not record["parse_error"] and not record["overflow"] for record in condition_records)})
display(pd.DataFrame(table_rows).set_index("Condition"))

### 3.3 Silently dropped relation labels
**Goal:** Count the schema-valid relations Llama emitted with labels outside the codebook (silently dropped at validation), and check whether the loss is systematic by the summary's source language (UNI-76).

In [ ]:
# Which out-of-codebook labels Llama invents (silently dropped at validation), per condition;
# then the drop rate per source language. A systematic per-language skew confirms UNI-76.
label_rows = []
for cond in RAW_CONDS:
    label_counts = Counter()
    n_malformed = n_bad_eid = 0
    for record in noise:
        if record["condition"] == cond:
            label_counts.update(record["dropped_labels"])
            n_malformed += record["n_drop_malformed"]
            n_bad_eid   += record["n_drop_eid"]
    n_kept            = sum(record["n_kept"] for record in noise if record["condition"] == cond)
    n_invalid_labels  = sum(label_counts.values())          # all out-of-codebook labels
    n_malformed_total = n_malformed + n_bad_eid             # label valid, entry structurally broken
    TOP_N = 8                                               # show the TOP_N most frequent labels; collapse the rest into one row
    top_labels = label_counts.most_common(TOP_N)
    for label, count in top_labels:
        label_rows.append({"Condition": cond, "Dropped as": f"invalid label `{label}`", "Count": count})
    n_other_invalid = n_invalid_labels - sum(count for _, count in top_labels)
    if n_other_invalid > 0:                                 # collapsed row so top-N + collapsed = total invalid-label
        label_rows.append({"Condition": cond,
                           "Dropped as": f"other invalid labels (collapsed, {len(label_counts) - len(top_labels)} types)",
                           "Count": n_other_invalid})
    label_rows.append({"Condition": cond, "Dropped as": "malformed entry / bad eID format",
                       "Count": n_malformed_total})
    n_total_dropped = n_invalid_labels + n_malformed_total  # every dropped row above (invalid labels + malformed)
    label_rows.append({"Condition": cond, "Dropped as": "TOTAL dropped",
                       "Count": f"{n_total_dropped:,} ({n_total_dropped / (n_total_dropped + n_kept):.2%} of emitted)"})
with pd.option_context("display.max_rows", None):   # show every invalid-label row (no truncation)
    display(pd.DataFrame(label_rows).set_index(["Condition", "Dropped as"]))

# Per source language (top 10 by row count): dropped / (kept + dropped).
top_languages = [lang for lang, _ in Counter(record["lang"] for record in noise).most_common(10)]
language_rows = []
for lang in top_languages:
    language_row = {"Lang": lang, "n summaries": sum(record["condition"] == RAW_CONDS[0] and record["lang"] == lang for record in noise)}
    for cond in RAW_CONDS:
        lang_condition_records = [record for record in noise if record["condition"] == cond and record["lang"] == lang]
        n_kept    = sum(record["n_kept"] for record in lang_condition_records)
        n_dropped = sum(sum(record["dropped_labels"].values()) for record in lang_condition_records)
        language_row[cond] = f"{n_dropped / (n_kept + n_dropped):.2%}" if (n_kept + n_dropped) else "—"
    language_rows.append(language_row)
display(pd.DataFrame(language_rows).set_index("Lang"))

### 3.4 Hallucinated event IDs
**Goal:** Report the rate of relations pointing at non-existent event IDs (dropped in §4.6) and the *e(N+1)* pattern behind them (UNI-60).

In [ ]:
# Get the hallucinated relations from the data
hallucinated_relations = [json.loads(l) for l in open(EXPERIMENT_DIR / "hallucinated_relations.jsonl", encoding="utf-8")]

# Create the table
table_rows = []
for cond in ["temporal", "causal", "temporal_causal_joint", "temporal_causal_independent"]:
    condition_hallucinations = [record for record in hallucinated_relations if record["condition"] == cond]
    n_kept = sum(len(relation_record["conditions"][cond]) for relation_record in rels)
    n_next_in_sequence = sum(bool(record.get("is_next_in_sequence")) for record in condition_hallucinations)
    table_rows.append({"Condition": cond, "Hallucinated": len(condition_hallucinations), "Kept": n_kept,
                       "Rate": f"{len(condition_hallucinations) / (len(condition_hallucinations) + n_kept):.2%}",
                       "% exactly e(N+1)": f"{n_next_in_sequence / len(condition_hallucinations):.1%}" if condition_hallucinations else "—"})
display(pd.DataFrame(table_rows).set_index("Condition"))

### 3.5 Pseudonymization trigger changes

**Goal:** Measure how much the event-trigger representation shifts under pseudonymization. Triggers are predicates rather than named entities, so the representation is anonymized by design, but the BERT+CRF extractor is re-run on the pseudonymized text and is context-sensitive, so a fraction of triggers still change. This quantifies that drift over the matched set and prints a few seed-fixed side-by-side exemplars.

**Note:** This needs the sibling pseudonymized run next to the non-pseudonymized one, named `anon_<EXPERIMENT_NAME>` in the same `data/experiments/` folder. The cell derives `ANON_DIR` from `EXPERIMENT_DIR`, so just make sure that run exists (run the pipeline's pseudonymized arm first if not).

In [ ]:
# Pseudonymization trigger drift: how much the event-trigger representation shifts when BERT+CRF is
# re-run on the pseudonymized text. Compares the matched set against the sibling anon_<experiment> run.
import json, re, collections, random, statistics, urllib.request, urllib.parse

ANON_DIR = EXPERIMENT_DIR.parent / ("anon_" + EXPERIMENT_DIR.name)   # sibling pseudonymized run
SEED, N_EXAMPLES, MAX_SENTS = 5, 2, 3
id_pattern = re.compile(r'"wikidata_id":\s*"([^"]*)",\s*"summary_id":\s*"([^"]*)"')

matched = {(m["wikidata_id"], m["summary_id"]) for m in meta}   # matched set, from Analysis 1

# Load the events for the matched summaries from one arm's processed-events file.
def load_events(events_path):
    events_by_key = {}
    with open(events_path, encoding="utf-8") as infile:
        for line in infile:
            match = id_pattern.search(line[:160])
            if match and (match.group(1), match.group(2)) in matched:
                record = json.loads(line)
                events_by_key[(match.group(1), match.group(2))] = {"sentences": record["sentences"], "events": record["events"]}
    return events_by_key

nonanon_events = load_events(EXPERIMENT_DIR / "tma_subset_events_processed.test.jsonl")
anon_events    = load_events(ANON_DIR       / "tma_subset_events_processed.test.jsonl")
shared_keys = [key for key in matched if key in nonanon_events and key in anon_events]

# Trigger set of a summary, and the Jaccard overlap between two trigger sets.
def trigger_set(summary):
    return {event["trigger"].lower().strip() for event in summary["events"]}
def jaccard(a, b):
    return len(a & b) / len(a | b) if (a | b) else 1.0

overlaps     = [jaccard(trigger_set(nonanon_events[key]), trigger_set(anon_events[key])) for key in shared_keys]
changed_keys = {key for key in shared_keys if trigger_set(nonanon_events[key]) != trigger_set(anon_events[key])}
print(f"Event-trigger drift across the matched set (n = {len(shared_keys)} summaries):")
print(f"  summaries whose trigger set changes under pseudonymization: {len(changed_keys)} ({100*len(changed_keys)/len(shared_keys):.1f}%)")
print(f"  mean trigger-set overlap (Jaccard), all summaries:          {100*statistics.mean(overlaps):.1f}%")
print(f"  mean trigger-set overlap, changed summaries only:           {100*statistics.mean([o for o, key in zip(overlaps, shared_keys) if key in changed_keys]):.1f}%\n")

# Render a summary's sentences with each trigger inlined as [trigger|event_type].
def inline_triggers(sentences, events):
    events_by_sentence = collections.defaultdict(list)
    for event in events: events_by_sentence[event["sent_id"]].append(event)
    rendered = []
    for sentence_idx, sentence in enumerate(sentences):
        cursor, pieces = 0, []
        for event in sorted(events_by_sentence.get(sentence_idx, []), key=lambda e: e["start"]):
            pieces += [sentence[cursor:event["start"]], f"[{event['trigger']}|{event['event_type']}]"]; cursor = event["end"]
        pieces.append(sentence[cursor:]); rendered.append("".join(pieces))
    return " ".join(rendered)

# Look up a work's English title from Wikidata (best-effort; falls back to the id).
def wikidata_title(wikidata_id):
    try:
        entity_id = "Q" + str(wikidata_id).lstrip("Q")
        url = "https://www.wikidata.org/w/api.php?" + urllib.parse.urlencode(
            {"action": "wbgetentities", "ids": entity_id, "props": "labels", "languages": "en", "format": "json"})
        request = urllib.request.Request(url, headers={"User-Agent": "uva-thesis-narrative/1.0 (research)"})
        with urllib.request.urlopen(request, timeout=20) as response:
            return json.load(response)["entities"][entity_id]["labels"].get("en", {}).get("value", entity_id)
    except Exception:
        return "Q" + str(wikidata_id)

candidates = [key for key in changed_keys if len(nonanon_events[key]["sentences"]) <= MAX_SENTS]
random.seed(SEED)
picks = random.sample(candidates, min(N_EXAMPLES, len(candidates)))
print(f"Exemplars (seed={SEED}, <= {MAX_SENTS} sentences, trigger set differs): {len(candidates)} candidates\n")
for key in picks:
    nonanon_triggers, anon_triggers = trigger_set(nonanon_events[key]), trigger_set(anon_events[key])
    display(Markdown(f"**{wikidata_title(key[0])}**  (`{key[0]}` / `{key[1]}`) - non-pseud-only triggers: "
                     f"`{sorted(nonanon_triggers - anon_triggers) or '-'}`; pseud-only: `{sorted(anon_triggers - nonanon_triggers) or '-'}`"))
    display(Markdown(f"*Non-pseudonymized:* {inline_triggers(nonanon_events[key]['sentences'], nonanon_events[key]['events'])}"))
    display(Markdown(f"*Pseudonymized:* {inline_triggers(anon_events[key]['sentences'], anon_events[key]['events'])}"))